In [1]:
"""
synthetic_dataloader.py
=======================
Builds loaders directly from a synthetic SIR simulation.
No EpiDataOrchestrator, no pipeline, no config.

Usage
-----
    from synthetic_dataloader import make_synthetic_loaders
    from src.utils.helpers import get_project_utilities_env
    import os

    graph_dir = os.path.join(get_project_utilities_env(), 'graphs', 'nuts3')

    geo_loader, lstm_loader = make_synthetic_loaders(
        shapefile     = data_orch.data_context.shapedata_node.dropna(),
        id_col        = data_orch.config.id_column,
        seq_len       = data_orch.config.sequence_length,
        horizon       = data_orch.config.horizon_leadtime,
        geo_graphname = 'geographical_neighbors1',
        graph_dir     = graph_dir,
        T             = 100,
    )

    gcn_model = GCNLSTMModel(dataloadermanager=geo_loader, name='gcn_synth')
    gcn_model.set_model_hparams()
    gcn_model.set_global_hparams(**global_hparams)
    gcn_model.train()
    gcn_model.forecast()
"""

from __future__ import annotations
import numpy as np
import geopandas as gpd
import torch
import os


def build_adjacency(shapefile: gpd.GeoDataFrame, id_col: str) -> dict:
    gdf    = shapefile[[id_col, 'geometry']].dropna().reset_index(drop=True)
    joined = gpd.sjoin(gdf, gdf, how='inner', predicate='touches')
    joined = joined[joined[f'{id_col}_left'] != joined[f'{id_col}_right']]
    node_ids = sorted(gdf[id_col].unique().tolist())
    adj = {nid: [] for nid in node_ids}
    for _, row in joined.iterrows():
        src, dst = int(row[f'{id_col}_left']), int(row[f'{id_col}_right'])
        if dst not in adj[src]:
            adj[src].append(dst)
    return adj


def simulate_sir(node_ids, adj, T=100, beta=0.4, gamma=0.05, seed=42):
    """
    Returns zscore-normalised incidence [T, N].
    Infection travels ONLY along adjacency edges.
    """
    rng = np.random.default_rng(seed)
    n   = len(node_ids)
    idx = {nid: i for i, nid in enumerate(node_ids)}

    S, I, R = np.ones(n), np.zeros(n), np.zeros(n)

    # seed the highest-degree node so wave propagates outward
    degrees   = {nid: len(nb) for nid, nb in adj.items()}
    seed_node = max(degrees, key=degrees.get)
    I[idx[seed_node]] = 0.05
    S[idx[seed_node]] = 0.95

    incidence = np.zeros((T, n))
    for t in range(T):
        incidence[t] = I.copy()
        dS, dI, dR = np.zeros(n), np.zeros(n), np.zeros(n)
        for nid in node_ids:
            i  = idx[nid]
            nb = adj.get(nid, [])
            nb_I = np.mean([I[idx[j]] for j in nb if j in idx]) if nb else 0.0
            new_inf = beta * nb_I * S[i]
            new_rec = gamma * I[i]
            dS[i] -= new_inf
            dI[i] += new_inf - new_rec
            dR[i] += new_rec
        S = np.clip(S + dS, 0, 1)
        I = np.clip(I + dI, 0, 1)
        R = np.clip(R + dR, 0, 1)

    mu, sigma = incidence.mean(), incidence.std() + 1e-8
    return (incidence - mu) / sigma


def make_sequences(incidence, seq_len, horizon, train_frac=0.6, val_frac=0.2):
    T, N = incidence.shape
    windows = []
    for t in range(T - seq_len - horizon + 1):
        X = torch.tensor(
            incidence[t:t+seq_len, :].T[:, np.newaxis, :],   # [N, 1, seq_len]
            dtype=torch.float
        )
        y = torch.tensor(
            incidence[t+seq_len+horizon-1, :, np.newaxis],    # [N, 1]
            dtype=torch.float
        )
        windows.append((X, y))
    n  = len(windows)
    t1 = int(n * train_frac)
    t2 = int(n * (train_frac + val_frac))
    return {'train': windows[:t1], 'val': windows[t1:t2],
            'test': windows[t2:], 'main': windows}


class SyntheticGraphDataLoaderManager:
    """Drop-in for GraphDataLoaderManager. Pass to GCNLSTMModel etc."""

    def __init__(self, incidence, edge_index, edge_weight,
                 seq_len, horizon, train_frac=0.6, val_frac=0.2):
        from src.dataloading.dataloaders.deepdataloaders.datacontainers import (
            GraphData, GraphDataList)

        splits = make_sequences(incidence, seq_len, horizon, train_frac, val_frac)

        def wrap(windows):
            return GraphDataList([
                GraphData(x=X, y=y, edge_index=edge_index, edge_weight=edge_weight)
                for X, y in windows
            ])

        self._dataloader_train = wrap(splits['train'])
        self._dataloader_val   = wrap(splits['val'])
        self._dataloader_test  = wrap(splits['test'])
        self._dataloader_main  = wrap(splits['main'])

    @property
    def dataloader_train(self): return self._dataloader_train
    @property
    def dataloader_val(self):   return self._dataloader_val
    @property
    def dataloader_test(self):  return self._dataloader_test
    @property
    def dataloader_main(self):  return self._dataloader_main


class SyntheticDeepDataLoaderManager:
    """Drop-in for DeepDataLoaderManager. Pass to LSTMModel etc."""

    def __init__(self, incidence, seq_len, horizon, train_frac=0.6, val_frac=0.2):
        from src.dataloading.dataloaders.deepdataloaders.datacontainers import (
            DeepData, DeepDataList)

        splits = make_sequences(incidence, seq_len, horizon, train_frac, val_frac)

        def wrap(windows):
            return DeepDataList([DeepData(x=X, y=y) for X, y in windows])

        self._dataloader_train = wrap(splits['train'])
        self._dataloader_val   = wrap(splits['val'])
        self._dataloader_test  = wrap(splits['test'])
        self._dataloader_main  = wrap(splits['main'])

    @property
    def dataloader_train(self): return self._dataloader_train
    @property
    def dataloader_val(self):   return self._dataloader_val
    @property
    def dataloader_test(self):  return self._dataloader_test
    @property
    def dataloader_main(self):  return self._dataloader_main


def make_synthetic_loaders(
    shapefile, id_col, seq_len, horizon, geo_graphname, graph_dir,
    T=100, beta=0.4, gamma=0.05, seed=42, train_frac=0.6, val_frac=0.2,
):
    """
    Returns (geo_loader, lstm_loader).
    Keep T short (80-120) so the wave peak falls in val/test, not only train.
    With beta=0.4, gamma=0.05, peak is around t=20-30 on a 400-node graph.
    train_frac=0.6 means train ends at t=54 on T=90 — wave peak well inside val.
    """
    node_ids = sorted(shapefile[id_col].dropna().unique().tolist())
    adj      = build_adjacency(shapefile, id_col)
    print(f"Nodes: {len(node_ids)}, avg neighbours: {np.mean([len(v) for v in adj.values()]):.1f}")

    incidence = simulate_sir(node_ids, adj, T=T, beta=beta, gamma=gamma, seed=seed)
    peak_t    = int(incidence.mean(axis=1).argmax())
    print(f"Peak at t={peak_t}/{T}  |  "
          f"train ends ~t{int(T*train_frac)}, "
          f"val ends ~t{int(T*(train_frac+val_frac))}")

    if peak_t < int(T * train_frac):
        print("WARNING: peak falls entirely in training. "
              "Increase T or reduce train_frac so val/test see the wave.")

    graph_path  = os.path.join(graph_dir, geo_graphname, geo_graphname)
    edge_index  = torch.load(graph_path + '_edge_index.pt', weights_only=False)
    edge_weight = torch.load(graph_path + '_edge_weight.pt', weights_only=False)
    print(f"Graph: {edge_index.shape[1]} edges")

    geo_loader  = SyntheticGraphDataLoaderManager(
        incidence, edge_index, edge_weight, seq_len, horizon, train_frac, val_frac)
    lstm_loader = SyntheticDeepDataLoaderManager(
        incidence, seq_len, horizon, train_frac, val_frac)

    print(f"Windows — train: {len(geo_loader.dataloader_train)}  "
          f"val: {len(geo_loader.dataloader_val)}  "
          f"test: {len(geo_loader.dataloader_test)}")

    return geo_loader, lstm_loader

In [ ]:

from src.utils.helpers import get_project_utilities_env
import os

from src.dataloading import EpiConfig, EpiDataOrchestrator
from src.dataloading import BaseLineDataLoaderManager
from src.models import PersistenceModel, ClimateologyModel, ConstantModel, ClimaScaleModel

epiconfig= EpiConfig.load_config('default_influenza_point_prediction')

data_orch= EpiDataOrchestrator(epiconfig).build()

graph_dir = os.path.join(get_project_utilities_env(), 'graphs', 'nuts3')

geo_loader, lstm_loader = make_synthetic_loaders(
    shapefile     = data_orch.data_context.shapedata_node.dropna(),
    id_col        = data_orch.config.id_column,
    seq_len       = data_orch.config.sequence_length,
    horizon       = data_orch.config.horizon_leadtime,
    geo_graphname = 'geographical_neighbors1',
    graph_dir     = graph_dir,
    T             = 90,          # short enough that wave spans all splits
    beta          = 0.4,
    gamma         = 0.05,
    train_frac    = 0.5,         # earlier cutoff so val/test see the peak
)

EpiConfig default_influenza_point_prediction loaded
Nodes: 400, avg neighbours: 5.2
Peak at t=52/90  |  train ends ~t45, val ends ~t62
Graph: 2488 edges
Windows — train: 42  val: 16  test: 26


In [5]:
geo_loader

In [4]:
from src.models import GCNLSTMModel

gcn_model = GCNLSTMModel(dataloadermanager=geo_loader, name='gcn_synth')
gcn_model.set_model_hparams()
gcn_model.set_global_hparams(**global_hparams)
gcn_model.train()
gcn_model.forecast()

# lstm_model = LSTMModel(dataloadermanager=lstm_loader, name='lstm_synth')
# lstm_model.set_model_hparams()
# lstm_model.set_global_hparams(**global_hparams)
# lstm_model.train()
# lstm_model.forecast()

AttributeError: 'SyntheticGraphDataLoaderManager' object has no attribute 'dataorchestrator'